In [2]:
import numpy as np

from keras_image_helper import create_preprocessor

In [3]:

def preprocess_pytorch(X):
    # X: shape (1, 299, 299, 3), dtype=float32, values in [0, 255]
    X = X / 255.0

    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

    # Convert NHWC → NCHW
    # from (batch, height, width, channels) → (batch, channels, height, width)
    X = X.transpose(0, 3, 1, 2)

    # Normalize
    X = (X - mean) / std

    return X.astype(np.float32)


preprocessor = create_preprocessor(preprocess_pytorch, target_size=(224, 224))

In [4]:
url = 'http://bit.ly/mlbookcamp-pants'
X = preprocessor.from_url(url)

In [14]:
X

array([[[[-0.14855723, -0.14855723, -0.11430772, ...,  0.00556555,
          -0.02868396, -0.18280673],
         [-0.13143247, -0.13143247, -0.08005822, ...,  0.00556555,
          -0.0115592 , -0.14855723],
         [-0.14855723, -0.14855723, -0.11430772, ...,  0.0226903 ,
          -0.0115592 , -0.11430772],
         ...,
         [-1.7411593 , -1.7582841 , -1.7240345 , ..., -1.34729   ,
          -1.34729   , -1.3815395 ],
         [-1.0732939 , -1.1246682 , -1.1760424 , ..., -1.278791  ,
          -1.278791  , -1.2959157 ],
         [-1.2274168 , -1.1760424 , -1.0904187 , ..., -1.210292  ,
          -1.1931672 , -1.2274168 ]],

        [[-0.10994396, -0.10994396, -0.07492995, ...,  0.13515407,
           0.10014006, -0.05742295],
         [-0.09243695, -0.09243695, -0.03991595, ...,  0.13515407,
           0.11764707, -0.02240895],
         [-0.10994396, -0.10994396, -0.07492995, ...,  0.15266107,
           0.11764707,  0.01260505],
         ...,
         [-1.7380952 , -1.7556022 

In [5]:
X.shape

(1, 3, 224, 224)

In [7]:
import onnxruntime as ort

onnx_model_path = "model_onnx/clothing_classifier.onnx"
session = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])

In [8]:
inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name

In [9]:
result = session.run([output_name], {input_name: X})

In [10]:
result

[array([[-0.6159841 , -9.704882  , -1.7348989 ,  0.30468893,  6.4438806 ,
         -1.1687253 , -8.077933  ,  1.0771865 , -5.4534383 , -4.4954805 ]],
       dtype=float32)]

In [11]:
predictions = result[0][0].tolist()

In [12]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

In [13]:
dict(zip(classes, predictions))

{'dress': -0.6159840822219849,
 'hat': -9.70488166809082,
 'longsleeve': -1.7348989248275757,
 'outwear': 0.3046889305114746,
 'pants': 6.443880558013916,
 'shirt': -1.1687252521514893,
 'shoes': -8.077933311462402,
 'shorts': 1.0771864652633667,
 'skirt': -5.4534382820129395,
 't-shirt': -4.495480537414551}